In [ ]:
import pandas as pd
import requests
import numpy as np
from tqdm import tqdm
from datetime import datetime

# --- Step 1: Load Metadata ---
metadata_path = "metadata_query_non_reservior_data_include_obs_inactive_20250529.csv"
metadata_df = pd.read_csv(metadata_path, dtype=str)
station_ids = metadata_df["STATION_ID"].dropna().unique().astype(str)

# --- Step 2: Download Discharge Data from API ---
print("🌐 Downloading discharge data from API...")
data_dict = {}
today = datetime.today().strftime("%Y-%m-%d")

for station_id in tqdm(station_ids, desc="Downloading", unit="station"):
    url = f"https://www.waterrights.utah.gov/dvrtdb/daily-chart.asp?station_id={station_id}&end_date={today}&f=json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            json_data = r.json()
            if "data" in json_data:
                df = pd.DataFrame(json_data["data"], columns=["date", "value"])
                df["date"] = pd.to_datetime(df["date"])
                df["value"] = pd.to_numeric(df["value"], errors="coerce")
                df = df.drop_duplicates(subset="date")
                df = df.set_index("date").rename(columns={"value": station_id})
                data_dict[station_id] = df
    except Exception as e:
        print(f"❌ Failed to download {station_id}: {e}")

# --- Step 3: Merge Time Series ---
print("📊 Merging all station data...")
merged_df = pd.concat(data_dict.values(), axis=1)

# --- Step 4: Compute Pearson Correlation ---
print("📈 Computing Pearson correlation matrix...")
corr_matrix = merged_df.corr(method="pearson", min_periods=30)

# --- Step 5: Filter Correlations ≥ 0.95 ---
print("🔍 Filtering highly correlated station pairs...")
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["Station1", "Station2", "Correlation"]
filtered_pairs = corr_pairs[corr_pairs["Correlation"] >= 0.95].copy()
print(f"✅ Found {len(filtered_pairs)} highly correlated pairs.")

# --- Step 6: Merge Metadata for Both Stations ---
print("🧩 Merging metadata for correlated stations...")
metadata_df["STATION_ID"] = metadata_df["STATION_ID"].astype(str)

merged1 = filtered_pairs.merge(metadata_df, left_on="Station1", right_on="STATION_ID", how="left").drop(columns=["STATION_ID"])
full_merged = merged1.merge(metadata_df, left_on="Station2", right_on="STATION_ID", how="left", suffixes=('_1', '_2')).drop(columns=["STATION_ID"])

# --- Step 7: Add Combined Station Page Link ---
full_merged["StationPage Combined"] = (
    "https://waterrights.utah.gov/dvrtdb/daily-chart.asp?STATION_ID="
    + full_merged["Station1"] + "," + full_merged["Station2"]
)

# --- Step 8: Reorder Columns with All Metadata Side-by-Side ---
core_columns = ['Station1', 'Station2', 'Correlation', 'StationPage Combined']
metadata_fields = [
    'COLLECTION_SYSTEM', 'collection_sys_description', 'MasterStationID', 'MasterStationName',
    'CollectionStationName', 'COMMENTS', 'SiteType', 'ANALOG_CHANNEL', 'SYSTEM_NAME',
    'DatasetType', 'MEASURING_DEVICE', 'DEVICE_TYPE', 'STATUS', 'LAT', 'LON', 'DataEntryMethod',
    'Telemetry', 'StationPage', 'UNITS_ID', 'RECORD_TYPE', 'UNITS_DESC_BASE', 'UNITS_DESC_ENTRY',
    'UNITS_MULTIPLIER', 'UNITS_DESC_REALTIME', 'NoOfYears', 'StartYr', 'EndYr'
]

ordered_columns = core_columns.copy()
for col in metadata_fields:
    ordered_columns.append(f"{col}_1")
    ordered_columns.append(f"{col}_2")

final_df = full_merged[ordered_columns]

# --- Step 9: Export Result ---
output_csv = "highly_correlated_stations_with_full_metadata_0.95.csv"
final_df.to_csv(output_csv, index=False)
print(f"📁 Output saved to: {output_csv}")